In [7]:
"""
Experimental Fluid Mechanics: Flow Determination via Drag on a Confined Sphere
Objective: Determine the mass flow rate of air in a tube based on the 
equilibrium of a confined sphere and compare it with the orifice plate method.

Libraries used:
- CoolProp: For high-accuracy fluid properties.
- uncertainties: For automatic error propagation.
"""

from CoolProp.CoolProp import PropsSI
from math import pi, e
from uncertainties import ufloat

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:
# Environmental Data (Table 1 from the report)
g = ufloat(9.78, 0.01)  # Local gravity [m/s²]
T_amb = ufloat(273 + 25.1, 0.5)  # Ambient temperature [K]
P_atm = ufloat(92525.7, 67)  # Ambient pressure [Pa]

# Fluid: Water (used for pressure measurement reference)
# Density calculation using CoolProp
rho_water = ufloat(PropsSI("D", "T", T_amb.nominal_value, "P", P_atm.nominal_value, 'water'), 0)

# Measured Experimental Pressures
p_static_h = ufloat(29.75/1000, 0.025/1000)  # Static pressure head [mH2O]
P_abs = P_atm + rho_water * g * p_static_h  # Absolute pressure [Pa]

# Working Fluid: Air Properties
fluid = 'air'
rho_air = ufloat(PropsSI("D", "T", T_amb.nominal_value, "P", P_abs.nominal_value, fluid), 0)
mu_air = ufloat(PropsSI("V", "T", T_amb.nominal_value, "P", P_abs.nominal_value, fluid), 0)

print(f"Air Density: {rho_air:.4f} kg/m³")
print(f"Dynamic Viscosity: {mu_air:.4e} Pa.s")

Air Density: 1.0850+/-0 kg/m³
Dynamic Viscosity: (1.8444+/-0)e-05 Pa.s


In [9]:
# Sphere Properties (Table 3)
m_sphere = ufloat(19.45/1000, 0.05/1000)  # Mass [kg]
d_sphere = ufloat(30.25/1000, 0.025/1000)  # Diameter [m]

# Calculated Sphere Geometry
a_frontal_sphere = (pi * d_sphere**2) / 4
vol_sphere = (4 * pi * (d_sphere/2)**3) / 3

# Tube Geometry
d_tube = ufloat(33.80/1000, 0.025/1000)  # Internal diameter [m]
a_tube = (pi * d_tube**2) / 4

# Confinement Ratio (d/D)
confinement_ratio = d_sphere / d_tube
print(f"Confinement Ratio: {confinement_ratio.nominal_value:.4f}")

Confinement Ratio: 0.8950


In [10]:
# Initial parameters for the iterative process
tol = 0.01
re_diff = 100
re_current = 100  # Initial guess for Reynolds number

while re_diff > tol:
    # Empirical Drag Coefficient correlation for spheres
    term1 = 24 / re_current
    term2 = (2.6 * (re_current / 5.0)) / (1 + (re_current / 5.0)**1.52)
    term3 = (0.411 * (re_current / (2.63e5))**-7.94) / (1 + (re_current / (2.63e5))**-8.0)
    term4 = (0.25 * (re_current / 1e6)) / (1 + (re_current / 1e6))
    cd_standard = term1 + term2 + term3 + term4

    # Flow velocity based on force equilibrium
    v_standard = (2 * g * ((m_sphere - rho_air * vol_sphere) / (a_frontal_sphere * cd_standard * rho_air)))**0.5
    
    # Update Reynolds number
    re_new = (rho_air * v_standard * d_tube) / mu_air
    re_diff = abs(re_new - re_current)
    re_current = re_new

# Final Mass Flow calculation
m_dot_standard = rho_air * v_standard * a_tube * 3600  # [kg/h]

print(f"Standard Velocity: {v_standard:.2f} m/s")
print(f"Standard Mass Flow: {m_dot_standard:.2f} kg/h")

C:\Users\andre\AppData\Local\Temp\ipykernel_37528\3428274414.py:19: FutureWarning: AffineScalarFunc.__abs__() is deprecated. It will be removed in a future release.
  re_diff = abs(re_new - re_current)
C:\Users\andre\AppData\Local\Temp\ipykernel_37528\3428274414.py:6: FutureWarning: AffineScalarFunc.__gt__() is deprecated. It will be removed in a future release.
  while re_diff > tol:


Standard Velocity: 34.32+/-0.05 m/s
Standard Mass Flow: 120.30+/-0.26 kg/h


In [11]:
# Resetting variables for the corrected model
re_diff = 100
re_current = 100
wall_correction_factor = 6.58  # Specific for d/D approx 0.89

while re_diff > tol:
    # Base Cd calculation
    cd_base = (24 / re_current) + \
              ((2.6 * (re_current / 5.0)) / (1 + (re_current / 5.0)**1.52)) + \
              (0.411 * (re_current / 2.63e5)**-7.94) / (1 + (re_current / 2.63e5)**-8.0) + \
              (0.25 * (re_current / 1e6)) / (1 + (re_current / 1e6))

    if re_current > 10000:
        cd_corrected = cd_base * (wall_correction_factor)**2
    else:
        cd_corrected = cd_base

    v_corrected = (2 * g * ((m_sphere - rho_air * vol_sphere) / (a_frontal_sphere * cd_corrected * rho_air)))**0.5
    re_new = (rho_air * v_corrected * d_tube) / mu_air
    re_diff = abs(re_new - re_current)
    re_current = re_new

m_dot_corrected = rho_air * v_corrected * a_tube * 3600  # [kg/h]

print(f"Corrected Velocity: {v_corrected:.3f} m/s")
print(f"Corrected Mass Flow: {m_dot_corrected:.2f} kg/h")

Corrected Velocity: 5.357+/-0.009 m/s
Corrected Mass Flow: 18.78+/-0.04 kg/h


C:\Users\andre\AppData\Local\Temp\ipykernel_37528\1870446110.py:20: FutureWarning: AffineScalarFunc.__abs__() is deprecated. It will be removed in a future release.
  re_diff = abs(re_new - re_current)
C:\Users\andre\AppData\Local\Temp\ipykernel_37528\1870446110.py:6: FutureWarning: AffineScalarFunc.__gt__() is deprecated. It will be removed in a future release.
  while re_diff > tol:
C:\Users\andre\AppData\Local\Temp\ipykernel_37528\1870446110.py:13: FutureWarning: AffineScalarFunc.__gt__() is deprecated. It will be removed in a future release.
  if re_current > 10000:


In [ ]:
# Reference value from Orifice Plate
m_dot_orifice = ufloat(26.58, 0.16)

print("--- FINAL COMPARISON ---")
print(f"Orifice Plate (Reference): {m_dot_orifice} kg/h")
print(f"Sphere Method (Corrected): {m_dot_corrected} kg/h")

error_percentage = abs(m_dot_corrected - m_dot_orifice) / m_dot_orifice * 100
print(f"Relative Deviation: {error_percentage.nominal_value:.2f}%")

--- FINAL COMPARISON ---
Orifice Plate (Reference): 26.58+/-0.16 kg/h
Sphere Method (Corrected): 18.78+/-0.04 kg/h
Relative Deviation: 29.36%


C:\Users\andre\AppData\Local\Temp\ipykernel_37528\2873373112.py:8: FutureWarning: AffineScalarFunc.__abs__() is deprecated. It will be removed in a future release.
  error_percentage = abs(m_dot_corrected - m_dot_orifice) / m_dot_orifice * 100
